# Music information retrieval & generation — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(z):
    z=np.asarray(z,dtype=float); e=np.exp(z-z.max()); return e/e.sum()
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float); return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def hz_to_mel(f): return 2595*np.log10(1+np.asarray(f)/700)
def mel_to_hz(m): return 700*(10**(np.asarray(m)/2595)-1)
def stft_mag(x,N=256,H=64):
    w=np.hanning(N); frames=[]
    for s in range(0,len(x)-N+1,H): frames.append(np.abs(np.fft.rfft(x[s:s+N]*w)))
    return np.array(frames).T
print('setup ok')

## Pitch, rhythm, and probabilistic music sequences
MIR reads musical structure from audio. These examples show chroma, tempo, octave invariance, next-note sampling, and a tiny melody representation.

In [ ]:
# 1) chroma folds a 440 Hz tone into pitch class 0
sr=8000; N=512; frame=np.sin(2*np.pi*440*np.arange(N)/sr); mag=np.abs(np.fft.rfft(frame*np.hanning(N))); chroma=np.zeros(12)
for k,m in enumerate(mag):
    f=k*sr/N
    if f>30: chroma[int(round(12*np.log2(f/440)))%12]+=m
plt.figure(figsize=(6,3)); plt.bar(np.arange(12),chroma/chroma.sum()); plt.xlabel('pitch class'); plt.title('A-like tone peaks at class 0')
assert int(np.argmax(chroma))==0 and abs(chroma[0]/chroma.sum()-0.475)<0.001

In [ ]:
# 2) tempo from evenly spaced beats
beats=np.array([0.0,0.5,1.0,1.5]); intervals=np.diff(beats); bpm=60/intervals.mean()
plt.figure(figsize=(6,2)); plt.vlines(beats,0,1); plt.yticks([]); plt.xlabel('seconds'); plt.title(f'tempo={bpm:.1f} BPM')
assert abs(bpm-120.0)<1e-9

In [ ]:
# 3) octave invariance maps 440 and 880 Hz to the same chroma class
freqs=np.array([440,880,660]); classes=np.round(12*np.log2(freqs/440)).astype(int)%12
plt.figure(figsize=(6,3)); plt.scatter(freqs,classes); plt.yticks(range(12)); plt.xlabel('Hz'); plt.ylabel('chroma class'); plt.title('octaves share a pitch class')
assert classes[0]==classes[1] and classes[2]==7

In [ ]:
# 4) generation samples from next-note probabilities
logits=np.array([2.0,1.0,0.0]); p=softmax(logits)
plt.figure(figsize=(6,3)); plt.bar(['note0','note1','note2'],p); plt.ylim(0,1); plt.title('next-note distribution')
assert np.allclose(np.round(p,3),[0.665,0.245,0.090]) and int(np.argmax(p))==0

In [ ]:
# 5) a tiny melody becomes a time-pitch piano-roll representation
notes=np.array([0,2,4,7,4,2,0]); roll=np.zeros((12,len(notes))); roll[notes,np.arange(len(notes))]=1
plt.figure(figsize=(6,3)); plt.imshow(roll,origin='lower',aspect='auto'); plt.xlabel('step'); plt.ylabel('pitch class'); plt.title('symbolic melody as sequence')
assert roll.sum()==len(notes) and notes[3]==7